## Backtrader example 3

This file deal with hourly EURUSD data over one year, in which we try to use Backtrader library to test a moving average crossover strategy

In this file we follow the following steps:
- Installing libraries
- Importing
- Fteching hourly EURUSD data  and cleaning 
- Defining strategy class with hourly timeframe adjustments
- Initialization and feeding data to  Backtrader's data format
- Run simulation
- Print Strategy Performance Metrics
- Plotting 

N.B. There is sill a problem when taking into account SL and TP 

### Importing libraries

In [22]:
import backtrader as bt
import backtrader.feeds as btfeeds
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

import datetime

%matplotlib inline

### Fteching hourly EURUSD data

In [23]:
EuroUsd= yf.Ticker("EURUSD=X")   #  import yfinance as yf and create ourselves a ticker object for a particular ticker (stock)
EuroUsd

yfinance.Ticker object <EURUSD=X>

In [24]:
# get 1hour historical data for EuroUsd between 02/06/2024 and 07/06/2024 (British format)
#  
EuroUsd_historical = EuroUsd.history(start="2024-01-01", end="2024-12-30", interval="1h")
EuroUsd_historical

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2024-01-01 18:00:00+00:00,1.105583,1.105583,1.105583,1.105583,0,0.0,0.0
2024-01-01 19:00:00+00:00,1.105583,1.105583,1.105339,1.105339,0,0.0,0.0
2024-01-01 20:00:00+00:00,1.105339,1.105339,1.103875,1.104240,0,0.0,0.0
2024-01-01 21:00:00+00:00,1.104240,1.105217,1.104240,1.105217,0,0.0,0.0
2024-01-01 22:00:00+00:00,1.104850,1.105094,1.104606,1.104728,0,0.0,0.0
...,...,...,...,...,...,...,...
2024-12-27 18:00:00+00:00,1.042427,1.043188,1.042427,1.042970,0,0.0,0.0
2024-12-27 19:00:00+00:00,1.042753,1.043297,1.042644,1.043079,0,0.0,0.0
2024-12-27 20:00:00+00:00,1.043079,1.043079,1.042535,1.042862,0,0.0,0.0


In [25]:
EuroUsd_historical = EuroUsd_historical.tz_localize(None)  # Remove timezone for Backtrader compatibility.info()

In [26]:
EuroUsd_historical

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2024-01-01 18:00:00,1.105583,1.105583,1.105583,1.105583,0,0.0,0.0
2024-01-01 19:00:00,1.105583,1.105583,1.105339,1.105339,0,0.0,0.0
2024-01-01 20:00:00,1.105339,1.105339,1.103875,1.104240,0,0.0,0.0
2024-01-01 21:00:00,1.104240,1.105217,1.104240,1.105217,0,0.0,0.0
2024-01-01 22:00:00,1.104850,1.105094,1.104606,1.104728,0,0.0,0.0
...,...,...,...,...,...,...,...
2024-12-27 18:00:00,1.042427,1.043188,1.042427,1.042970,0,0.0,0.0
2024-12-27 19:00:00,1.042753,1.043297,1.042644,1.043079,0,0.0,0.0
2024-12-27 20:00:00,1.043079,1.043079,1.042535,1.042862,0,0.0,0.0


In [27]:
cols = ['Dividends','Stock Splits']   
(EuroUsd_historical[cols] == 0).all()
EuroUsd_historical.drop(columns= cols, inplace = True)
EuroUsd_historical.columns = EuroUsd_historical.columns.str.lower()  # Add this line
EuroUsd_historical

,open,high,low,close,volume
Datetime,,,,,
2024-01-01 18:00:00,1.105583,1.105583,1.105583,1.105583,0
2024-01-01 19:00:00,1.105583,1.105583,1.105339,1.105339,0
2024-01-01 20:00:00,1.105339,1.105339,1.103875,1.104240,0
2024-01-01 21:00:00,1.104240,1.105217,1.104240,1.105217,0
2024-01-01 22:00:00,1.104850,1.105094,1.104606,1.104728,0
...,...,...,...,...,...
2024-12-27 18:00:00,1.042427,1.043188,1.042427,1.042970,0
2024-12-27 19:00:00,1.042753,1.043297,1.042644,1.043079,0
2024-12-27 20:00:00,1.043079,1.043079,1.042535,1.042862,0


In [28]:
df=EuroUsd_historical.copy()

In [29]:
# Defining strategy class with hourly timeframe adjustments
class HourlyMACrossover(bt.Strategy):
    params = (
        ('fast_ma', 14),    # 50-hour MA
        ('slow_ma', 50),   # 200-hour MA
        ('risk_per_trade', 0.01),  # 1% risk per trade
        ('stop_pct', 0.005),       # 0.5% stop loss
        ('reward_ratio', 2),       # 2:1 reward/risk
    )

    def __init__(self):
        self.dataclose = self.datas[0].close
        #self.fast_ma = bt.indicators.SMA(self.data.close, period=self.p.fast_ma)
        self.fast_ma= bt.indicators.MovingAverageSimple(self.datas[0], period=self.params.fast_ma)
        #self.slow_ma = bt.indicators.SMA(self.data.close, period=self.p.slow_ma)
        self.slow_ma= bt.indicators.MovingAverageSimple(self.datas[0], period=self.params.slow_ma)
        
        self.order = None
        self.trade_count = 0
        self.win_count = 0

    def log(self, txt):
        dt = self.datas[0].datetime.datetime(0)
        print(f'{dt}: {txt}')

    def notify_trade(self, trade):
        if trade.isclosed:
            self.trade_count += 1
            if trade.pnl > 0:
                self.win_count += 1

    def next(self):
        if self.order:
            return

        if not self.position:
            if self.fast_ma[0] > self.slow_ma[0] and self.fast_ma[-1] <= self.slow_ma[-1]:
                risk_amount = self.broker.getvalue() * self.p.risk_per_trade
                price = self.data.close[0]
                stop_price = price * (1 - self.p.stop_pct)
                position_size = risk_amount / abs(price - stop_price)
                
                take_profit = price * (1 + self.p.stop_pct * self.p.reward_ratio)
                self.order = self.buy_bracket(size=position_size,
                                              exectype=bt.Order.Market,
                                              stopprice=stop_price,
                                              price=price,
                                              limitprice=take_profit)
                print(f"Buy at {self.data.close[0]}")
        else:
            if self.fast_ma[0] < self.slow_ma[0] and self.fast_ma[-1] >= self.slow_ma[-1]:
                self.close()
                print(f"Sell at {self.data.close[0]}")

### Initialization and feeding data to  Backtrader's data format

In [30]:
cerebro = bt.Cerebro()

In [31]:
# Parse/Convert to Backtrader's PandasData format
bt_data = bt.feeds.PandasData(dataname=df,
                              #timeframe=bt.TimeFrame.Minutes,
                              #compression=60,  # 60 minutes = hourly
                              datetime=None,   # Use index
                              open=0,
                              high=1,
                              low=2,
                              close=3,
                              volume=4,
                              openinterest=None
                              )

In [32]:

cerebro.adddata(bt_data)
cerebro.addstrategy(HourlyMACrossover)
cerebro.broker.setcash(100000.0)
cerebro.broker.setcommission(commission=0.0002)
cerebro.addsizer(bt.sizers.FixedSize, stake=10000)

In [33]:
# Add analyzers for performance metrics
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')  # Sharpe Ratio
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')     # Total and annual returns
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')   # Drawdown stats
cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trades') # Trade statistics

### Run simulation

In [34]:
# Run the backtest
results = cerebro.run()
strat = results[0]  # Get the first (and only) strategy instance

Buy at 1.09505033493042


In [35]:
strat

### Print Strategy Performance Metrics

In [36]:
# Extract and print performance metrics
print("=== Strategy Performance ===")
# Starting and Final Portfolio Value
start_value = cerebro.broker.startingcash
final_value = cerebro.broker.getvalue()
print(f"Starting Portfolio Value: ${start_value:.2f}")
print(f"Final Portfolio Value: ${final_value:.2f}")
print(f"Profit/Loss: ${(final_value - start_value):.2f}")

=== Strategy Performance ===
Starting Portfolio Value: $100000.00
Final Portfolio Value: $100000.00
Profit/Loss: $0.00


In [37]:
# Returns
total_return = strat.analyzers.returns.get_analysis()['rtot'] * 100  # Total return as percentage
annual_return = strat.analyzers.returns.get_analysis()['rnorm'] * 100  # Annualized return
print(f"Total Return: {total_return:.2f}%")
print(f"Annualized Return: {annual_return:.2f}%")

Total Return: 0.00%
Annualized Return: 0.00%


In [38]:
# Sharpe Ratio
sharpe_ratio = strat.analyzers.sharpe.get_analysis()['sharperatio']
print(f"Sharpe Ratio: {sharpe_ratio:.2f}" if sharpe_ratio is not None else "Sharpe Ratio: N/A")

# Drawdown
max_drawdown = strat.analyzers.drawdown.get_analysis()['max']['drawdown']
print(f"Max Drawdown: {max_drawdown:.2f}%")

Sharpe Ratio: N/A
Max Drawdown: 0.00%


In [39]:
# Trade Statistics
trade_analysis = strat.analyzers.trades.get_analysis()
total_trades = trade_analysis.get('total', {}).get('total', 0)
won_trades = trade_analysis.get('won', {}).get('total', 0)
lost_trades = trade_analysis.get('lost', {}).get('total', 0)
win_rate = (won_trades / total_trades * 100) if total_trades > 0 else 0
avg_profit = trade_analysis.get('pnl', {}).get('net', {}).get('average', 0)
print(f"Total Trades: {total_trades}")
print(f"Won Trades: {won_trades}")
print(f"Lost Trades: {lost_trades}")
print(f"Win Rate: {win_rate:.2f}%")
print(f"Average Profit per Trade: ${avg_profit:.2f}")

Total Trades: 0
Won Trades: 0
Lost Trades: 0
Win Rate: 0.00%
Average Profit per Trade: $0.00


In [40]:
# cerebro.plot(iplot=False)